In [2]:
import pandas as pd
import numpy as np

# Load datasets again
ball_df = pd.read_csv('/content/sample_data/deliveries.csv')
match_df = pd.read_csv('/content/sample_data/matches.csv')

# Standardize column names
ball_df.columns = ball_df.columns.str.lower().str.strip()
match_df.columns = match_df.columns.str.lower().str.strip()

# Merge match info
match_df = match_df[['id', 'venue', 'season', 'date']]
match_df.rename(columns={'id': 'match_id'}, inplace=True)

ball_df = ball_df.merge(match_df, on='match_id', how='left')

print("ball_df loaded successfully:", ball_df.shape)

batsman_match = ball_df.groupby(
    ['match_id', 'batter', 'venue', 'season', 'date']
).agg(
    runs=('batsman_runs', 'sum'),
    balls=('ball', 'count')
).reset_index()

batsman_match['strike_rate'] = (batsman_match['runs'] / batsman_match['balls']) * 100
batsman_match.head()

batsman_match['date'] = pd.to_datetime(batsman_match['date'])
batsman_match = batsman_match.sort_values(['batter', 'date'])

batsman_match['avg_runs_last_5'] = (
    batsman_match.groupby('batter')['runs']
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(0, drop=True)
)

batsman_match['avg_runs_last_10'] = (
    batsman_match.groupby('batter')['runs']
    .rolling(10, min_periods=1)
    .mean()
    .reset_index(0, drop=True)
)

venue_avg = batsman_match.groupby(['batter', 'venue'])['runs'].mean().reset_index()
venue_avg.rename(columns={'runs': 'venue_avg_runs'}, inplace=True)

batsman_match = batsman_match.merge(
    venue_avg, on=['batter', 'venue'], how='left'
)
batsman_match['career_avg_runs'] = (
    batsman_match.groupby('batter')['runs']
    .expanding()
    .mean()
    .reset_index(0, drop=True)
)
batsman_match['target_runs'] = (
    batsman_match.groupby('batter')['runs']
    .shift(-1)
)

batsman_match = batsman_match.dropna()

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = batsman_match[
    ['avg_runs_last_5', 'avg_runs_last_10', 'career_avg_runs', 'venue']
]
y = batsman_match['target_runs']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['avg_runs_last_5', 'avg_runs_last_10', 'career_avg_runs']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['venue'])
    ]
)

batsman_match.to_csv('dataset.csv', index=False)


ball_df loaded successfully: (260920, 20)
